# Imports

In [49]:
#install Vader through "pip install vaderSentiment" in terminal
#install NLTK through "pip install user -U nltk"
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
import pandas as pd
import nltk
from scipy import stats
nltk.download('punkt')  # Required for TextBlob tokenization
nltk.download('averaged_perceptron_tagger')  # Required for TextBlob
from textblob import TextBlob
from scipy import stats
import numpy as np
df = pd.read_csv('../../Data/2. IntermediateData/df_binarized_hard.csv')

[nltk_data] Downloading package punkt to /Users/brookeye/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /Users/brookeye/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


In [35]:
df

,utterance_id,conversation_id,text,speaker,movie_id,reply_to,speaker_id,character_name,movie_id.1,title,...,credit_pos,year,id,imdbid,bechdel_score,imdb_score,numVotes,runtimeMinutes,genres,oscar
0,L1045,L1044,They do not!,u0,m0,L1044,u0,BIANCA,m0,10 things i hate about you,...,4,1999,374,147800,1,7.4,424659,97,"Comedy,Drama,Romance",0
1,L1044,L1044,They do to!,u2,m0,NaN,u2,CAMERON,m0,10 things i hate about you,...,3,1999,374,147800,1,7.4,424659,97,"Comedy,Drama,Romance",0
2,L985,L984,I hope so.,u0,m0,L984,u0,BIANCA,m0,10 things i hate about you,...,4,1999,374,147800,1,7.4,424659,97,"Comedy,Drama,Romance",0
3,L984,L984,She okay?,u2,m0,NaN,u2,CAMERON,m0,10 things i hate about you,...,3,1999,374,147800,1,7.4,424659,97,"Comedy,Drama,Romance",0
4,L925,L924,Let's go.,u0,m0,L924,u0,BIANCA,m0,10 things i hate about you,...,4,1999,374,147800,1,7.4,424659,97,"Comedy,Drama,Romance",0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
137057,L665991,L665987,"I'm sorry, sir. We only seat by reservation.",u9021,m615,L665990,u9021,MAITRE D',m615,young frankenstein,...,?,1974,1077,72431,0,8.0,176404,106,Comedy,0
137058,L665990,L665987,Food!!,u9023,m615,L665989,u9023,MONSTER,m615,young frankenstein,...,?,1974,1077,72431,0,8.0,176404,106,Comedy,0
137059,L665989,L665987,Do you have a reservation?,u9021,m615,L665988,u9021,MAITRE D',m615,young frankenstein,...,?,1974,1077,72431,0,8.0,176404,106,Comedy,0
137060,L665988,L665987,Food!,u9023,m615,L665987,u9023,MONSTER,m615,young frankenstein,...,?,1974,1077,72431,0,8.0,176404,106,Comedy,0


In [36]:
df=df.drop(['movie_id.1'], axis=1)

# Textblob (exploratory). Conclusion: Unreliable

# Calculating sentiment score with VADER (DF_Hard)

In [37]:
# Initialize the sentiment analyzer
analyzer = SentimentIntensityAnalyzer()

# Function to get polarity scores
def get_sentiment(text):
    return analyzer.polarity_scores(text)['compound']

# Apply sentiment analysis to each line of dialogue
df['sentiment'] = df['text'].apply(get_sentiment)

# 1. Mean sentiment polarity by gender overall
mean_sentiment_by_gender = df.groupby('gender')['sentiment'].mean()

# 2. Mean sentiment polarity by gender split by Bechdel score
mean_sentiment_by_gender_bechdel = df.groupby(['bechdel_score', 'gender'])['sentiment'].mean()

# To present it more clearly, you can unstack the results
mean_sentiment_split = mean_sentiment_by_gender_bechdel.unstack()

print("Overall mean sentiment by gender:")
print(mean_sentiment_by_gender)

print("\nMean sentiment by gender and Bechdel score:")
print(mean_sentiment_split)

Overall mean sentiment by gender:
gender
f    0.058220
m    0.048871
Name: sentiment, dtype: float64

Mean sentiment by gender and Bechdel score:
gender                f         m
bechdel_score                    
0              0.058090  0.044079
1              0.058289  0.055099


In [38]:
df

,utterance_id,conversation_id,text,speaker,movie_id,reply_to,speaker_id,character_name,title,gender,...,year,id,imdbid,bechdel_score,imdb_score,numVotes,runtimeMinutes,genres,oscar,sentiment
0,L1045,L1044,They do not!,u0,m0,L1044,u0,BIANCA,10 things i hate about you,f,...,1999,374,147800,1,7.4,424659,97,"Comedy,Drama,Romance",0,0.0000
1,L1044,L1044,They do to!,u2,m0,NaN,u2,CAMERON,10 things i hate about you,m,...,1999,374,147800,1,7.4,424659,97,"Comedy,Drama,Romance",0,0.0000
2,L985,L984,I hope so.,u0,m0,L984,u0,BIANCA,10 things i hate about you,f,...,1999,374,147800,1,7.4,424659,97,"Comedy,Drama,Romance",0,0.4404
3,L984,L984,She okay?,u2,m0,NaN,u2,CAMERON,10 things i hate about you,m,...,1999,374,147800,1,7.4,424659,97,"Comedy,Drama,Romance",0,0.2263
4,L925,L924,Let's go.,u0,m0,L924,u0,BIANCA,10 things i hate about you,f,...,1999,374,147800,1,7.4,424659,97,"Comedy,Drama,Romance",0,0.0000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
137057,L665991,L665987,"I'm sorry, sir. We only seat by reservation.",u9021,m615,L665990,u9021,MAITRE D',young frankenstein,f,...,1974,1077,72431,0,8.0,176404,106,Comedy,0,-0.0772
137058,L665990,L665987,Food!!,u9023,m615,L665989,u9023,MONSTER,young frankenstein,m,...,1974,1077,72431,0,8.0,176404,106,Comedy,0,0.0000
137059,L665989,L665987,Do you have a reservation?,u9021,m615,L665988,u9021,MAITRE D',young frankenstein,f,...,1974,1077,72431,0,8.0,176404,106,Comedy,0,0.0000
137060,L665988,L665987,Food!,u9023,m615,L665987,u9023,MONSTER,young frankenstein,m,...,1974,1077,72431,0,8.0,176404,106,Comedy,0,0.0000


In [51]:
# Initialize analyzer
analyzer = SentimentIntensityAnalyzer()
df['sentiment'] = df['text'].apply(lambda x: analyzer.polarity_scores(x)['compound'])

# Function for Cohen's d
def cohens_d(group1, group2):
    n1, n2 = len(group1), len(group2)
    pooled_std = np.sqrt(((n1-1)*group1.std()**2 + (n2-1)*group2.std()**2) / (n1 + n2 - 2))
    return abs((group1.mean() - group2.mean()) / pooled_std)

# ===== 1. Gender Differences =====
male_all = df[df['gender'] == 'm']['sentiment']
female_all = df[df['gender'] == 'f']['sentiment']
t_gender, p_gender = stats.ttest_ind(female_all, male_all, equal_var=False)

print("="*50)
print("GENDER DIFFERENCES (OVERALL)")
print(f"Female mean: {female_all.mean():.3f} | Male mean: {male_all.mean():.3f}")
print(f"t = {t_gender:.2f}, p = {p_gender:.10f} {'***' if p_gender < 0.001 else '**' if p_gender < 0.01 else '*' if p_gender < 0.05 else 'ns'}")
print(f"Effect size (Cohen's d): {cohens_d(female_all, male_all):.2f}")

# ===== 2. Gender Differences by Bechdel =====
print("\n" + "="*50)
print("GENDER DIFFERENCES BY BECHDEL SCORE")

for bechdel in [0, 1]:
    female = df[(df['gender']=='f') & (df['bechdel_score']==bechdel)]['sentiment']
    male = df[(df['gender']=='m') & (df['bechdel_score']==bechdel)]['sentiment']
    t, p = stats.ttest_ind(female, male, equal_var=False)
    
    print(f"\nBECHDEL {bechdel}:")
    print(f"Female mean: {female.mean():.3f} | Male mean: {male.mean():.3f}")
    print(f"t = {t:.2f}, p = {p:.10f} {'***' if p < 0.000000001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'}")
    print(f"Effect size: {cohens_d(female, male):.2f}")

# ===== 3. Bechdel Differences by Gender =====
print("\n" + "="*50)
print("BECHDEL DIFFERENCES BY GENDER")

for gender in ['f', 'm']:
    bechdel1 = df[(df['gender']==gender) & (df['bechdel_score']==1)]['sentiment']
    bechdel0 = df[(df['gender']==gender) & (df['bechdel_score']==0)]['sentiment']
    t, p = stats.ttest_ind(bechdel1, bechdel0, equal_var=False)
    
    print(f"\n{gender.upper()} DIALOGUE:")
    print(f"Bechdel 1 mean: {bechdel1.mean():.3f} | Bechdel 0 mean: {bechdel0.mean():.3f}")
    print(f"t = {t:.2f}, p = {p:.10f} {'***' if p < 0.00000000001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'}")
    print(f"Effect size: {cohens_d(bechdel1, bechdel0):.2f}")

# ===== 4. Sample Sizes =====
print("\n" + "="*50)
print("SAMPLE SIZES")
print(f"Total female: {len(female_all)} | Total male: {len(male_all)}")
print("\nBy Bechdel Score:")
for bechdel in [0, 1]:
    print(f"Bechdel {bechdel} - Female: {len(df[(df['gender']=='f') & (df['bechdel_score']==bechdel)])} | Male: {len(df[(df['gender']=='m') & (df['bechdel_score']==bechdel)])}")

GENDER DIFFERENCES (OVERALL)
Female mean: 0.058 | Male mean: 0.049
t = 4.72, p = 0.0000023399 ***
Effect size (Cohen's d): 0.03

GENDER DIFFERENCES BY BECHDEL SCORE

BECHDEL 0:
Female mean: 0.058 | Male mean: 0.044
t = 4.51, p = 0.0000065357 **
Effect size: 0.04

BECHDEL 1:
Female mean: 0.058 | Male mean: 0.055
t = 1.19, p = 0.2322788612 ns
Effect size: 0.01

BECHDEL DIFFERENCES BY GENDER

F DIALOGUE:
Bechdel 1 mean: 0.058 | Bechdel 0 mean: 0.058
t = 0.06, p = 0.9531287151 ns
Effect size: 0.00

M DIALOGUE:
Bechdel 1 mean: 0.055 | Bechdel 0 mean: 0.044
t = 4.77, p = 0.0000018184 **
Effect size: 0.03

SAMPLE SIZES
Total female: 44695 | Total male: 92367

By Bechdel Score:
Bechdel 0 - Female: 15459 | Male: 52201
Bechdel 1 - Female: 29236 | Male: 40166


### Trying to do intensity (aboluste value). Results inconclusive

In [43]:
# Initialize the sentiment analyzer
analyzer = SentimentIntensityAnalyzer()

# Function to get ABSOLUTE polarity scores (intensity)
def get_sentiment_intensity(text):
    return abs(analyzer.polarity_scores(text)['compound'])  # Absolute value for intensity

# Apply sentiment intensity analysis to each line of dialogue
df['sentiment_intensity'] = df['text'].apply(get_sentiment_intensity)

# 1. Mean sentiment intensity by gender overall
mean_intensity_by_gender = df.groupby('gender')['sentiment_intensity'].mean()

# 2. Mean sentiment intensity by gender split by Bechdel score
mean_intensity_by_gender_bechdel = df.groupby(['bechdel_score', 'gender'])['sentiment_intensity'].mean()

# Unstack for clearer presentation
mean_intensity_split = mean_intensity_by_gender_bechdel.unstack()

print("Overall mean sentiment INTENSITY (absolute polarity) by gender:")
print(mean_intensity_by_gender)

print("\nMean sentiment INTENSITY by gender and Bechdel score:")
print(mean_intensity_split)

Overall mean sentiment INTENSITY (absolute polarity) by gender:
gender
f    0.224141
m    0.225896
Name: sentiment_intensity, dtype: float64

Mean sentiment INTENSITY by gender and Bechdel score:
gender                f         m
bechdel_score                    
0              0.219504  0.221201
1              0.226593  0.231998


In [44]:
df

,utterance_id,conversation_id,text,speaker,movie_id,reply_to,speaker_id,character_name,title,gender,...,bechdel_score,imdb_score,numVotes,runtimeMinutes,genres,oscar,sentiment,sentiment_category,sentiment_textblob,sentiment_intensity
0,L1045,L1044,They do not!,u0,m0,L1044,u0,BIANCA,10 things i hate about you,f,...,1,7.4,424659,97,"Comedy,Drama,Romance",0,0.0000,Neutral,0.00,0.0000
1,L1044,L1044,They do to!,u2,m0,NaN,u2,CAMERON,10 things i hate about you,m,...,1,7.4,424659,97,"Comedy,Drama,Romance",0,0.0000,Neutral,0.00,0.0000
2,L985,L984,I hope so.,u0,m0,L984,u0,BIANCA,10 things i hate about you,f,...,1,7.4,424659,97,"Comedy,Drama,Romance",0,0.4404,Neutral,0.00,0.4404
3,L984,L984,She okay?,u2,m0,NaN,u2,CAMERON,10 things i hate about you,m,...,1,7.4,424659,97,"Comedy,Drama,Romance",0,0.2263,Positive,0.50,0.2263
4,L925,L924,Let's go.,u0,m0,L924,u0,BIANCA,10 things i hate about you,f,...,1,7.4,424659,97,"Comedy,Drama,Romance",0,0.0000,Neutral,0.00,0.0000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
137057,L665991,L665987,"I'm sorry, sir. We only seat by reservation.",u9021,m615,L665990,u9021,MAITRE D',young frankenstein,f,...,0,8.0,176404,106,Comedy,0,-0.0772,Negative,-0.25,0.0772
137058,L665990,L665987,Food!!,u9023,m615,L665989,u9023,MONSTER,young frankenstein,m,...,0,8.0,176404,106,Comedy,0,0.0000,Neutral,0.00,0.0000
137059,L665989,L665987,Do you have a reservation?,u9021,m615,L665988,u9021,MAITRE D',young frankenstein,f,...,0,8.0,176404,106,Comedy,0,0.0000,Neutral,0.00,0.0000
137060,L665988,L665987,Food!,u9023,m615,L665987,u9023,MONSTER,young frankenstein,m,...,0,8.0,176404,106,Comedy,0,0.0000,Neutral,0.00,0.0000


In [40]:

def get_sentiment_textblob(text):
    return TextBlob(text).sentiment.polarity  # Range: [-1, 1]

df['sentiment_textblob'] = df['text'].apply(get_sentiment_textblob)

# Compute means
print("\nOverall mean sentiment by gender (TextBlob):")
print(df.groupby('gender')['sentiment_textblob'].mean())

print("\nMean sentiment by Bechdel score and gender (TextBlob):")
print(df.groupby(['bechdel_score', 'gender'])['sentiment_textblob'].mean().unstack())


Overall mean sentiment by gender (TextBlob):
gender
f    0.039871
m    0.044765
Name: sentiment_textblob, dtype: float64

Mean sentiment by Bechdel score and gender (TextBlob):
gender                f         m
bechdel_score                    
0              0.041194  0.042875
1              0.039172  0.047223


In [42]:
# Categorize sentiment first
df['sentiment_category'] = df['sentiment_textblob'].apply(
    lambda x: 'Positive' if x >= 0.05 
    else 'Negative' if x <= -0.05 
    else 'Neutral'
)

# Print text for each category
print("="*50 + "\nFIRST 40 POSITIVE TEXTS\n" + "="*50)
for text in df[df['sentiment_category'] == 'Positive']['text'].head(200):
    print(text)
    print("-"*50)  # Separator between entries

# print("\n" + "="*50 + "\nFIRST 40 NEUTRAL TEXTS\n" + "="*50) 
# for text in df[df['sentiment_category'] == 'Neutral']['text'].head(40):
#     print(text)
#     print("-"*50)

print("\n" + "="*50 + "\nFIRST 40 NEGATIVE TEXTS\n" + "="*50)
for text in df[df['sentiment_category'] == 'Negative']['text'].head(200):
    print(text)
    print("-"*50)

FIRST 40 POSITIVE TEXTS
She okay?
--------------------------------------------------
Wow
--------------------------------------------------
Okay -- you're gonna need to learn how to lie.
--------------------------------------------------
The "real you".
--------------------------------------------------
What good stuff?
--------------------------------------------------
I figured you'd get to the good stuff eventually.
--------------------------------------------------
Thank God!  If I had to hear one more story about your coiffure...
--------------------------------------------------
Have fun tonight?
--------------------------------------------------
So that's the kind of guy she likes? Pretty ones?
--------------------------------------------------
Lesbian?  No. I found a picture of Jared Leto in one of her drawers, so I'm pretty sure she's not harboring same-sex tendencies.
--------------------------------------------------
I really, really, really wanna go, but I can't.  Not unles

# DF_Medium

In [54]:
df = pd.read_csv('../../Data/2. IntermediateData/df_binarized_medium.csv')
# Initialize the sentiment analyzer
analyzer = SentimentIntensityAnalyzer()

# Function to get polarity scores
def get_sentiment(text):
    return analyzer.polarity_scores(text)['compound']

# Apply sentiment analysis to each line of dialogue
df['sentiment'] = df['text'].apply(get_sentiment)

# 1. Mean sentiment polarity by gender overall
mean_sentiment_by_gender = df.groupby('gender')['sentiment'].mean()

# 2. Mean sentiment polarity by gender split by Bechdel score
mean_sentiment_by_gender_bechdel = df.groupby(['bechdel_score', 'gender'])['sentiment'].mean()

# To present it more clearly, you can unstack the results
mean_sentiment_split = mean_sentiment_by_gender_bechdel.unstack()

print("Overall mean sentiment by gender:")
print(mean_sentiment_by_gender)

print("\nMean sentiment by gender and Bechdel score:")
print(mean_sentiment_split)

# Initialize analyzer
analyzer = SentimentIntensityAnalyzer()
df['sentiment'] = df['text'].apply(lambda x: analyzer.polarity_scores(x)['compound'])

# Function for Cohen's d
def cohens_d(group1, group2):
    n1, n2 = len(group1), len(group2)
    pooled_std = np.sqrt(((n1-1)*group1.std()**2 + (n2-1)*group2.std()**2) / (n1 + n2 - 2))
    return abs((group1.mean() - group2.mean()) / pooled_std)

# ===== 1. Gender Differences =====
male_all = df[df['gender'] == 'm']['sentiment']
female_all = df[df['gender'] == 'f']['sentiment']
t_gender, p_gender = stats.ttest_ind(female_all, male_all, equal_var=False)

print("="*50)
print("GENDER DIFFERENCES (OVERALL)")
print(f"Female mean: {female_all.mean():.3f} | Male mean: {male_all.mean():.3f}")
print(f"t = {t_gender:.2f}, p = {p_gender:.10f} {'***' if p_gender < 0.001 else '**' if p_gender < 0.01 else '*' if p_gender < 0.05 else 'ns'}")
print(f"Effect size (Cohen's d): {cohens_d(female_all, male_all):.2f}")

# ===== 2. Gender Differences by Bechdel =====
print("\n" + "="*50)
print("GENDER DIFFERENCES BY BECHDEL SCORE")

for bechdel in [0, 1]:
    female = df[(df['gender']=='f') & (df['bechdel_score']==bechdel)]['sentiment']
    male = df[(df['gender']=='m') & (df['bechdel_score']==bechdel)]['sentiment']
    t, p = stats.ttest_ind(female, male, equal_var=False)
    
    print(f"\nBECHDEL {bechdel}:")
    print(f"Female mean: {female.mean():.3f} | Male mean: {male.mean():.3f}")
    print(f"t = {t:.2f}, p = {p:.10f} {'***' if p < 0.000000001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'}")
    print(f"Effect size: {cohens_d(female, male):.2f}")

# ===== 3. Bechdel Differences by Gender =====
print("\n" + "="*50)
print("BECHDEL DIFFERENCES BY GENDER")

for gender in ['f', 'm']:
    bechdel1 = df[(df['gender']==gender) & (df['bechdel_score']==1)]['sentiment']
    bechdel0 = df[(df['gender']==gender) & (df['bechdel_score']==0)]['sentiment']
    t, p = stats.ttest_ind(bechdel1, bechdel0, equal_var=False)
    
    print(f"\n{gender.upper()} DIALOGUE:")
    print(f"Bechdel 1 mean: {bechdel1.mean():.3f} | Bechdel 0 mean: {bechdel0.mean():.3f}")
    print(f"t = {t:.2f}, p = {p:.15f} {'***' if p < 0.00000000001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'}")
    print(f"Effect size: {cohens_d(bechdel1, bechdel0):.2f}")

# ===== 4. Sample Sizes =====
print("\n" + "="*50)
print("SAMPLE SIZES")
print(f"Total female: {len(female_all)} | Total male: {len(male_all)}")
print("\nBy Bechdel Score:")
for bechdel in [0, 1]:
    print(f"Bechdel {bechdel} - Female: {len(df[(df['gender']=='f') & (df['bechdel_score']==bechdel)])} | Male: {len(df[(df['gender']=='m') & (df['bechdel_score']==bechdel)])}")

Overall mean sentiment by gender:
gender
f    0.058220
m    0.048871
Name: sentiment, dtype: float64

Mean sentiment by gender and Bechdel score:
gender                f         m
bechdel_score                    
0              0.045992  0.035129
1              0.061502  0.058284
GENDER DIFFERENCES (OVERALL)
Female mean: 0.058 | Male mean: 0.049
t = 4.72, p = 0.0000023399 ***
Effect size (Cohen's d): 0.03

GENDER DIFFERENCES BY BECHDEL SCORE

BECHDEL 0:
Female mean: 0.046 | Male mean: 0.035
t = 2.82, p = 0.0048484681 **
Effect size: 0.03

BECHDEL 1:
Female mean: 0.062 | Male mean: 0.058
t = 1.36, p = 0.1739440047 ns
Effect size: 0.01

BECHDEL DIFFERENCES BY GENDER

F DIALOGUE:
Bechdel 1 mean: 0.062 | Bechdel 0 mean: 0.046
t = 3.99, p = 0.000067503417468 **
Effect size: 0.05

M DIALOGUE:
Bechdel 1 mean: 0.058 | Bechdel 0 mean: 0.035
t = 10.03, p = 0.000000000000000 ***
Effect size: 0.07

SAMPLE SIZES
Total female: 44695 | Total male: 92367

By Bechdel Score:
Bechdel 0 - Female: 9457 | 

# DF_Easy

In [53]:
df = pd.read_csv('../../Data/2. IntermediateData/df_binarized_easy.csv')
# Initialize the sentiment analyzer
analyzer = SentimentIntensityAnalyzer()

# Function to get polarity scores
def get_sentiment(text):
    return analyzer.polarity_scores(text)['compound']

# Apply sentiment analysis to each line of dialogue
df['sentiment'] = df['text'].apply(get_sentiment)

# 1. Mean sentiment polarity by gender overall
mean_sentiment_by_gender = df.groupby('gender')['sentiment'].mean()

# 2. Mean sentiment polarity by gender split by Bechdel score
mean_sentiment_by_gender_bechdel = df.groupby(['bechdel_score', 'gender'])['sentiment'].mean()

# To present it more clearly, you can unstack the results
mean_sentiment_split = mean_sentiment_by_gender_bechdel.unstack()

print("Overall mean sentiment by gender:")
print(mean_sentiment_by_gender)

print("\nMean sentiment by gender and Bechdel score:")
print(mean_sentiment_split)

# Initialize analyzer
analyzer = SentimentIntensityAnalyzer()
df['sentiment'] = df['text'].apply(lambda x: analyzer.polarity_scores(x)['compound'])

# Function for Cohen's d
def cohens_d(group1, group2):
    n1, n2 = len(group1), len(group2)
    pooled_std = np.sqrt(((n1-1)*group1.std()**2 + (n2-1)*group2.std()**2) / (n1 + n2 - 2))
    return abs((group1.mean() - group2.mean()) / pooled_std)

# ===== 1. Gender Differences =====
male_all = df[df['gender'] == 'm']['sentiment']
female_all = df[df['gender'] == 'f']['sentiment']
t_gender, p_gender = stats.ttest_ind(female_all, male_all, equal_var=False)

print("="*50)
print("GENDER DIFFERENCES (OVERALL)")
print(f"Female mean: {female_all.mean():.3f} | Male mean: {male_all.mean():.3f}")
print(f"t = {t_gender:.2f}, p = {p_gender:.10f} {'***' if p_gender < 0.001 else '**' if p_gender < 0.01 else '*' if p_gender < 0.05 else 'ns'}")
print(f"Effect size (Cohen's d): {cohens_d(female_all, male_all):.2f}")

# ===== 2. Gender Differences by Bechdel =====
print("\n" + "="*50)
print("GENDER DIFFERENCES BY BECHDEL SCORE")

for bechdel in [0, 1]:
    female = df[(df['gender']=='f') & (df['bechdel_score']==bechdel)]['sentiment']
    male = df[(df['gender']=='m') & (df['bechdel_score']==bechdel)]['sentiment']
    t, p = stats.ttest_ind(female, male, equal_var=False)
    
    print(f"\nBECHDEL {bechdel}:")
    print(f"Female mean: {female.mean():.3f} | Male mean: {male.mean():.3f}")
    print(f"t = {t:.2f}, p = {p:.10f} {'***' if p < 0.000000001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'}")
    print(f"Effect size: {cohens_d(female, male):.2f}")

# ===== 3. Bechdel Differences by Gender =====
print("\n" + "="*50)
print("BECHDEL DIFFERENCES BY GENDER")

for gender in ['f', 'm']:
    bechdel1 = df[(df['gender']==gender) & (df['bechdel_score']==1)]['sentiment']
    bechdel0 = df[(df['gender']==gender) & (df['bechdel_score']==0)]['sentiment']
    t, p = stats.ttest_ind(bechdel1, bechdel0, equal_var=False)
    
    print(f"\n{gender.upper()} DIALOGUE:")
    print(f"Bechdel 1 mean: {bechdel1.mean():.3f} | Bechdel 0 mean: {bechdel0.mean():.3f}")
    print(f"t = {t:.2f}, p = {p:.10f} {'***' if p < 0.00000000001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'}")
    print(f"Effect size: {cohens_d(bechdel1, bechdel0):.2f}")

# ===== 4. Sample Sizes =====
print("\n" + "="*50)
print("SAMPLE SIZES")
print(f"Total female: {len(female_all)} | Total male: {len(male_all)}")
print("\nBy Bechdel Score:")
for bechdel in [0, 1]:
    print(f"Bechdel {bechdel} - Female: {len(df[(df['gender']=='f') & (df['bechdel_score']==bechdel)])} | Male: {len(df[(df['gender']=='m') & (df['bechdel_score']==bechdel)])}")

Overall mean sentiment by gender:
gender
f    0.058220
m    0.048871
Name: sentiment, dtype: float64

Mean sentiment by gender and Bechdel score:
gender                f         m
bechdel_score                    
0              0.028641  0.023745
1              0.059243  0.051453
GENDER DIFFERENCES (OVERALL)
Female mean: 0.058 | Male mean: 0.049
t = 4.72, p = 0.0000023399 ***
Effect size (Cohen's d): 0.03

GENDER DIFFERENCES BY BECHDEL SCORE

BECHDEL 0:
Female mean: 0.029 | Male mean: 0.024
t = 0.52, p = 0.6052688914 ns
Effect size: 0.01

BECHDEL 1:
Female mean: 0.059 | Male mean: 0.051
t = 3.82, p = 0.0001315327 **
Effect size: 0.02

BECHDEL DIFFERENCES BY GENDER

F DIALOGUE:
Bechdel 1 mean: 0.059 | Bechdel 0 mean: 0.029
t = 3.44, p = 0.0005880963 **
Effect size: 0.09

M DIALOGUE:
Bechdel 1 mean: 0.051 | Bechdel 0 mean: 0.024
t = 7.17, p = 0.0000000000 ***
Effect size: 0.08

SAMPLE SIZES
Total female: 44695 | Total male: 92367

By Bechdel Score:
Bechdel 0 - Female: 1494 | Male: 8607
